# 04 — Feature Engineering (PySpark)

## Objective
Transform raw multi-table data into a single modeling-ready
dataset — one row per customer, 40-50 features — using
PySpark aggregations, window functions, and SparkSQL.

## The Core Transformation
Input:
  customers    : 50,000 rows × 19 columns
  transactions : 16,608,362 rows × 11 columns
  products     : 124,865 rows × 7 columns
  clv_labels   : 50,000 rows × 5 columns

Output:
  modeling_data.csv : 50,000 rows × ~45 features

## Feature Categories
1. Static customer features (direct from customers table)
2. Transaction aggregation features (groupBy)
3. Rolling window features (3-month vs 9-month spend)
4. Product holding features (pivot from products table)
5. Engineered ratio features (formulas)
6. Encoding (one-hot, target encoding)

## Key New PySpark Concept
rowsBetween() window for rolling aggregations —
the most powerful feature engineering tool in PySpark.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DateType, IntegerType, DoubleType
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"]        = r"C:\hadoop\bin;" + os.environ["PATH"]

JAR_PATH   = os.path.abspath("../jars/sqlite-jdbc-3.45.1.0.jar")
DB_PATH    = r"D:\some\other\drive\finance_clv.db"
DB_URL     = f"jdbc:sqlite:{DB_PATH}"
PROCESSED  = "../data/processed"
LOCAL_TEMP = os.path.abspath("../data/spark_temp")
os.makedirs(LOCAL_TEMP, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("CLV_Feature_Engineering")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.extraClassPath", JAR_PATH)
    .config("spark.executor.extraClassPath", JAR_PATH)
    .config("spark.local.dir", LOCAL_TEMP)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# Load cleaned CSVs
customers = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv(f"{PROCESSED}/customers_clean.csv")

products  = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv(f"{PROCESSED}/products_clean.csv")

clv       = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv(f"{PROCESSED}/clv_labels_clean.csv")

txn       = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv(f"{PROCESSED}/transactions_clean.csv")

# Fix date types
customers = customers.withColumn(
    "account_open_date",
    F.to_date(F.col("account_open_date"), "yyyy-MM-dd")
)
txn = txn.withColumn(
    "txn_date",
    F.to_date(F.col("txn_date"), "yyyy-MM-dd")
).withColumn(
    "txn_month", F.month("txn_date")
).withColumn(
    "txn_year",  F.year("txn_date")
)

# Register views
customers.createOrReplaceTempView("customers")
products.createOrReplaceTempView("products")
clv.createOrReplaceTempView("clv_labels")
txn.createOrReplaceTempView("transactions")

# Cache transactions — used in every feature category
txn.cache()
_ = txn.count()

print("Data loaded ✅")
print(f"  customers    : {customers.count():>10,}")
print(f"  products     : {products.count():>10,}")
print(f"  clv_labels   : {clv.count():>10,}")
print(f"  transactions : {txn.count():>10,} (cached)")

Data loaded ✅
  customers    :     50,000
  products     :    124,865
  clv_labels   :     50,000
  transactions : 16,608,362 (cached)


In [2]:
print("CATEGORY 1 — STATIC CUSTOMER FEATURES\n")
print("These come directly from the customers table.")
print("No aggregation needed — one row per customer already.\n")

# Select and rename the features we need
# Drop columns that are identifiers or leakage risks
static_features = customers.select(
    "customer_id",
    "segment",
    "city_tier",
    "city",
    "age",
    "gender",
    "monthly_income",
    "cibil_score",
    "digital_score",
    "account_age_months",
    "kyc_complete",
    "pan_linked",
    "aadhaar_linked",
    "rm_assigned",
    "is_nri",
    "is_prime_driver"
    if "is_prime_driver" in customers.columns
    else F.lit(False).alias("is_prime_driver"),
)

# Cast boolean columns to integer for modeling
bool_cols = [
    "kyc_complete","pan_linked","aadhaar_linked",
    "rm_assigned","is_nri"
]
for col in bool_cols:
    if col in static_features.columns:
        static_features = static_features.withColumn(
            col, F.col(col).cast(IntegerType())
        )

print(f"Static features shape: {static_features.count():,} rows "
      f"× {len(static_features.columns)} columns")
print(f"\nColumns: {static_features.columns}")

# Quick validation — no nulls in key columns
print("\nNull check on key static features:")
static_features.select([
    F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(c)
    for c in ["monthly_income","cibil_score",
              "digital_score","account_age_months"]
]).show()

CATEGORY 1 — STATIC CUSTOMER FEATURES

These come directly from the customers table.
No aggregation needed — one row per customer already.

Static features shape: 50,000 rows × 16 columns

Columns: ['customer_id', 'segment', 'city_tier', 'city', 'age', 'gender', 'monthly_income', 'cibil_score', 'digital_score', 'account_age_months', 'kyc_complete', 'pan_linked', 'aadhaar_linked', 'rm_assigned', 'is_nri', 'is_prime_driver']

Null check on key static features:
+--------------+-----------+-------------+------------------+
|monthly_income|cibil_score|digital_score|account_age_months|
+--------------+-----------+-------------+------------------+
|             0|          0|            0|                 0|
+--------------+-----------+-------------+------------------+



In [3]:
print("CATEGORY 2 — TRANSACTION AGGREGATION FEATURES\n")
print("Aggregating 16.6M rows → 50K customer-level features\n")

txn_features = spark.sql("""
    SELECT
        customer_id,

        -- Volume features
        COUNT(txn_id)                                   AS total_txns,
        ROUND(COUNT(txn_id) / 12.0, 2)                 AS avg_monthly_txns,

        -- Monetary features (debit only = actual spending)
        ROUND(SUM(CASE WHEN credit_debit = 'DR'
                       THEN amount ELSE 0 END), 2)      AS total_annual_debit,
        ROUND(AVG(CASE WHEN credit_debit = 'DR'
                       THEN amount END), 2)             AS avg_debit_amount,
        ROUND(AVG(amount), 2)                           AS avg_txn_amount,
        ROUND(STDDEV(amount), 2)                        AS std_txn_amount,
        ROUND(MAX(amount), 2)                           AS max_txn_amount,
        ROUND(MIN(CASE WHEN credit_debit = 'DR'
                       THEN amount END), 2)             AS min_debit_amount,

        -- Channel features
        ROUND(SUM(CASE WHEN txn_type = 'UPI'
                       THEN 1 ELSE 0 END) * 100.0 /
              COUNT(*), 2)                              AS upi_pct,
        ROUND(SUM(CASE WHEN txn_type IN ('NEFT','RTGS')
                       THEN 1 ELSE 0 END) * 100.0 /
              COUNT(*), 2)                              AS wire_transfer_pct,
        ROUND(SUM(CASE WHEN txn_type = 'ATM'
                       THEN 1 ELSE 0 END) * 100.0 /
              COUNT(*), 2)                              AS atm_pct,
        ROUND(SUM(CASE WHEN txn_type = 'Branch'
                       THEN 1 ELSE 0 END) * 100.0 /
              COUNT(*), 2)                              AS branch_pct,

        -- Behavioral features
        ROUND(SUM(CASE WHEN is_international = 1
                       THEN 1 ELSE 0 END) * 100.0 /
              COUNT(*), 2)                              AS intl_txn_pct,
        COUNT(DISTINCT merchant_category)               AS unique_categories,
        COUNT(DISTINCT txn_type)                        AS unique_txn_types,
        COUNT(DISTINCT txn_month)                       AS active_months,

        -- Weekend spending
        ROUND(SUM(CASE WHEN txn_month IN (11, 12)
                       THEN amount ELSE 0 END), 2)      AS q4_spend,
        ROUND(SUM(CASE WHEN txn_month IN (1, 2, 3)
                       THEN amount ELSE 0 END), 2)      AS q1_spend

    FROM transactions
    GROUP BY customer_id
""")

txn_features.createOrReplaceTempView("txn_features")

print(f"Transaction features: {txn_features.count():,} rows "
      f"× {len(txn_features.columns)} columns")

print("\nSample:")
txn_features.show(3, truncate=False)

print("\nNull check:")
txn_features.select([
    F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(c)
    for c in ["total_txns","avg_txn_amount",
              "std_txn_amount","total_annual_debit"]
]).show()


CATEGORY 2 — TRANSACTION AGGREGATION FEATURES

Aggregating 16.6M rows → 50K customer-level features

Transaction features: 50,000 rows × 19 columns

Sample:
+-----------+----------+----------------+------------------+----------------+--------------+--------------+--------------+----------------+-------+-----------------+-------+----------+------------+-----------------+----------------+-------------+----------+------------+
|customer_id|total_txns|avg_monthly_txns|total_annual_debit|avg_debit_amount|avg_txn_amount|std_txn_amount|max_txn_amount|min_debit_amount|upi_pct|wire_transfer_pct|atm_pct|branch_pct|intl_txn_pct|unique_categories|unique_txn_types|active_months|q4_spend  |q1_spend    |
+-----------+----------+----------------+------------------+----------------+--------------+--------------+--------------+----------------+-------+-----------------+-------+----------+------------+-----------------+----------------+-------------+----------+------------+
|CUST000014 |176       |14.67 

In [4]:
print("CATEGORY 3 — ROLLING WINDOW FEATURES\n")
print("Most important new concept in Feature Engineering.")
print("3-month spend vs 9-month spend — is customer growing?\n")

# Step 1 — Monthly aggregation per customer
monthly_debit = spark.sql("""
    SELECT
        customer_id,
        txn_year,
        txn_month,
        ROUND(SUM(CASE WHEN credit_debit = 'DR'
                       THEN amount ELSE 0 END), 2) AS monthly_debit,
        COUNT(txn_id)                               AS monthly_txns
    FROM transactions
    GROUP BY customer_id, txn_year, txn_month
""")

monthly_debit.createOrReplaceTempView("monthly_debit")

# Step 2 — Define window: partition by customer, order by time
w = Window.partitionBy("customer_id") \
          .orderBy("txn_year", "txn_month")

# Step 3 — Compute rolling features
monthly_with_rolling = monthly_debit.withColumn(
    # Cumulative spend up to current month
    "cumulative_debit",
    F.round(
        F.sum("monthly_debit").over(
            w.rowsBetween(Window.unboundedPreceding, 0)
        ), 2
    )
).withColumn(
    # Rolling 3-month spend (current + 2 previous months)
    "rolling_3m_spend",
    F.round(
        F.sum("monthly_debit").over(
            w.rowsBetween(-2, 0)
        ), 2
    )
).withColumn(
    # Month-over-month change
    "prev_month_debit",
    F.lag("monthly_debit", 1).over(w)
).withColumn(
    "mom_change",
    F.round(
        F.col("monthly_debit") - F.col("prev_month_debit"), 2
    )
)

monthly_with_rolling.createOrReplaceTempView("monthly_rolling")

# Step 4 — Extract last 3 months vs first 9 months
# Last 3 months = October, November, December (months 10,11,12)
# First 9 months = January through September (months 1-9)
window_features = spark.sql("""
    SELECT
        customer_id,
        ROUND(SUM(CASE WHEN txn_month >= 10
                       THEN monthly_debit ELSE 0 END), 2)
                                            AS spend_last_3m,
        ROUND(SUM(CASE WHEN txn_month < 10
                       THEN monthly_debit ELSE 0 END), 2)
                                            AS spend_first_9m,
        ROUND(AVG(monthly_debit), 2)        AS avg_monthly_debit,
        ROUND(STDDEV(monthly_debit), 2)     AS std_monthly_debit,
        MAX(monthly_debit)                  AS peak_month_spend,
        MIN(monthly_debit)                  AS min_month_spend,
        COUNT(CASE WHEN monthly_debit = 0
                   THEN 1 END)              AS zero_spend_months
    FROM monthly_debit
    GROUP BY customer_id
""")

# Compute spend ratio: last 3m vs first 9m (normalized)
# Ratio > 1 means customer spending more recently = growing
# Ratio < 1 means customer spending less recently = declining
window_features = window_features.withColumn(
    "spend_trend_ratio",
    F.round(
        (F.col("spend_last_3m") / 3) /
        (F.col("spend_first_9m") / 9 + F.lit(1.0)),
        4
    )
)

window_features.createOrReplaceTempView("window_features")

print(f"Window features: {window_features.count():,} rows "
      f"× {len(window_features.columns)} columns")

print("\nSpend trend ratio distribution:")
window_features.select(
    F.min("spend_trend_ratio").alias("min"),
    F.percentile_approx("spend_trend_ratio",0.25,100).alias("p25"),
    F.percentile_approx("spend_trend_ratio",0.50,100).alias("median"),
    F.percentile_approx("spend_trend_ratio",0.75,100).alias("p75"),
    F.max("spend_trend_ratio").alias("max")
).show()

print("\nSample window features:")
window_features.show(5, truncate=False)

CATEGORY 3 — ROLLING WINDOW FEATURES

Most important new concept in Feature Engineering.
3-month spend vs 9-month spend — is customer growing?



Window features: 50,000 rows × 9 columns

Spend trend ratio distribution:
+---+------+------+------+-------+
|min|   p25|median|   p75|    max|
+---+------+------+------+-------+
|0.0|0.5717|0.9051|1.3958|24.8355|
+---+------+------+------+-------+


Sample window features:
+-----------+-------------+--------------+-----------------+-----------------+----------------+---------------+-----------------+-----------------+
|customer_id|spend_last_3m|spend_first_9m|avg_monthly_debit|std_monthly_debit|peak_month_spend|min_month_spend|zero_spend_months|spend_trend_ratio|
+-----------+-------------+--------------+-----------------+-----------------+----------------+---------------+-----------------+-----------------+
|CUST000019 |1298098.78   |4110210.61    |450692.45        |347963.99        |1133490.31      |116747.07      |0                |0.9475           |
|CUST000025 |372292.71    |1431616.66    |150325.78        |102938.88        |352501.3        |42337.2        |0                |0.78

In [5]:
print("CATEGORY 4 — PRODUCT HOLDING FEATURES\n")
print("Pivoting product table from long format to wide format.\n")

# Step 1 — Product count and active count per customer
product_summary = spark.sql("""
    SELECT
        customer_id,
        COUNT(product_id)                            AS total_products,
        SUM(CASE WHEN is_active = true
                 THEN 1 ELSE 0 END)                  AS active_products,

        -- Binary flags for each product type
        MAX(CASE WHEN product_type = 'credit_card'
                      AND is_active = true
                 THEN 1 ELSE 0 END)                  AS has_credit_card,
        MAX(CASE WHEN product_type = 'home_loan'
                      AND is_active = true
                 THEN 1 ELSE 0 END)                  AS has_home_loan,
        MAX(CASE WHEN product_type = 'personal_loan'
                      AND is_active = true
                 THEN 1 ELSE 0 END)                  AS has_personal_loan,
        MAX(CASE WHEN product_type = 'mutual_fund'
                      AND is_active = true
                 THEN 1 ELSE 0 END)                  AS has_mutual_fund,
        MAX(CASE WHEN product_type = 'fixed_deposit'
                      AND is_active = true
                 THEN 1 ELSE 0 END)                  AS has_fd,
        MAX(CASE WHEN product_type = 'insurance'
                      AND is_active = true
                 THEN 1 ELSE 0 END)                  AS has_insurance,

        -- Product values
        ROUND(SUM(CASE WHEN product_type = 'home_loan'
                       THEN current_value ELSE 0 END), 2)
                                                     AS home_loan_value,
        ROUND(SUM(CASE WHEN product_type = 'personal_loan'
                       THEN current_value ELSE 0 END), 2)
                                                     AS personal_loan_value,
        ROUND(SUM(CASE WHEN product_type = 'credit_card'
                       THEN current_value ELSE 0 END), 2)
                                                     AS credit_card_limit,
        ROUND(SUM(CASE WHEN product_type = 'mutual_fund'
                       THEN current_value ELSE 0 END), 2)
                                                     AS mutual_fund_value,
        ROUND(SUM(CASE WHEN product_type = 'fixed_deposit'
                       THEN current_value ELSE 0 END), 2)
                                                     AS fd_value,
        ROUND(SUM(CASE WHEN is_active = true
                       THEN current_value ELSE 0 END), 2)
                                                     AS total_portfolio_value
    FROM products
    GROUP BY customer_id
""")

product_summary.createOrReplaceTempView("product_summary")

print(f"Product features: {product_summary.count():,} rows "
      f"× {len(product_summary.columns)} columns")

print("\nProduct holding rates:")
product_summary.select(
    F.round(F.avg("total_products"),2).alias("avg_products"),
    F.round(F.avg("active_products"),2).alias("avg_active"),
    F.round(F.avg("has_credit_card")*100,1).alias("cc_pct"),
    F.round(F.avg("has_home_loan")*100,1).alias("hl_pct"),
    F.round(F.avg("has_mutual_fund")*100,1).alias("mf_pct"),
    F.round(F.avg("has_fd")*100,1).alias("fd_pct"),
    F.round(F.avg("has_insurance")*100,1).alias("ins_pct"),
).show()

print("\nPortfolio value distribution:")
product_summary.select(
    F.round(F.avg("total_portfolio_value"),2).alias("avg_portfolio"),
    F.round(F.avg("home_loan_value"),2).alias("avg_home_loan"),
    F.round(F.avg("fd_value"),2).alias("avg_fd"),
    F.round(F.avg("mutual_fund_value"),2).alias("avg_mf"),
).show()

CATEGORY 4 — PRODUCT HOLDING FEATURES

Pivoting product table from long format to wide format.

Product features: 50,000 rows × 15 columns

Product holding rates:
+------------+----------+------+------+------+------+-------+
|avg_products|avg_active|cc_pct|hl_pct|mf_pct|fd_pct|ins_pct|
+------------+----------+------+------+------+------+-------+
|         2.5|      2.35|  34.4|   8.1|  22.7|  23.6|   20.7|
+------------+----------+------+------+------+------+-------+


Portfolio value distribution:
+-------------+-------------+----------+----------+
|avg_portfolio|avg_home_loan|    avg_fd|    avg_mf|
+-------------+-------------+----------+----------+
|   5051242.69|   1466765.56|2019858.07|1171322.54|
+-------------+-------------+----------+----------+



In [6]:
print("ASSEMBLING FINAL MODELING DATASET\n")

static_features.createOrReplaceTempView("static_features_view")

modeling_data = spark.sql("""
    SELECT
        s.customer_id, s.segment, s.city_tier, s.city,
        s.age, s.gender, s.monthly_income, s.cibil_score,
        s.digital_score, s.account_age_months,
        s.kyc_complete, s.pan_linked, s.aadhaar_linked,
        s.rm_assigned, s.is_nri,
        t.total_txns, t.avg_monthly_txns, t.total_annual_debit,
        t.avg_debit_amount, t.avg_txn_amount, t.std_txn_amount,
        t.max_txn_amount, t.upi_pct, t.wire_transfer_pct,
        t.atm_pct, t.intl_txn_pct, t.unique_categories,
        t.unique_txn_types, t.active_months,
        t.q4_spend, t.q1_spend,
        w.spend_last_3m, w.spend_first_9m,
        w.spend_trend_ratio, w.avg_monthly_debit,
        w.std_monthly_debit, w.peak_month_spend,
        w.zero_spend_months,
        p.total_products, p.active_products,
        p.has_credit_card, p.has_home_loan,
        p.has_personal_loan, p.has_mutual_fund,
        p.has_fd, p.has_insurance,
        p.home_loan_value,
        p.personal_loan_value,
        p.fd_value, p.mutual_fund_value,
        p.credit_card_limit, p.total_portfolio_value,
        l.clv_next_12months, l.clv_bucket
    FROM static_features_view s
    JOIN txn_features     t ON s.customer_id = t.customer_id
    JOIN window_features  w ON s.customer_id = w.customer_id
    JOIN product_summary  p ON s.customer_id = p.customer_id
    JOIN clv_labels       l ON s.customer_id = l.customer_id
""")

modeling_data.createOrReplaceTempView("modeling_data")

print(f"Modeling dataset: {modeling_data.count():,} rows "
      f"× {len(modeling_data.columns)} columns")

print("\nNull check across all numeric features:")
null_counts = modeling_data.select([
    F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(c)
    for c in modeling_data.columns
    if c not in ["customer_id","segment","city_tier",
                 "city","gender","clv_bucket"]
]).collect()[0]

has_nulls = {k: v for k, v in null_counts.asDict().items() if v > 0}
if has_nulls:
    print(f"  Columns with nulls: {has_nulls}")
else:
    print("  No nulls found ✅")

print(f"\nColumn list:")
for i, c in enumerate(modeling_data.columns, 1):
    print(f"  {i:>2}. {c}")

ASSEMBLING FINAL MODELING DATASET



Modeling dataset: 50,000 rows × 54 columns

Null check across all numeric features:
  Columns with nulls: {'kyc_complete': 1516}

Column list:
   1. customer_id
   2. segment
   3. city_tier
   4. city
   5. age
   6. gender
   7. monthly_income
   8. cibil_score
   9. digital_score
  10. account_age_months
  11. kyc_complete
  12. pan_linked
  13. aadhaar_linked
  14. rm_assigned
  15. is_nri
  16. total_txns
  17. avg_monthly_txns
  18. total_annual_debit
  19. avg_debit_amount
  20. avg_txn_amount
  21. std_txn_amount
  22. max_txn_amount
  23. upi_pct
  24. wire_transfer_pct
  25. atm_pct
  26. intl_txn_pct
  27. unique_categories
  28. unique_txn_types
  29. active_months
  30. q4_spend
  31. q1_spend
  32. spend_last_3m
  33. spend_first_9m
  34. spend_trend_ratio
  35. avg_monthly_debit
  36. std_monthly_debit
  37. peak_month_spend
  38. zero_spend_months
  39. total_products
  40. active_products
  41. has_credit_card
  42. has_home_loan
  43. has_personal_loan
  44. has_mutua

In [7]:
# Fix the 1,516 kyc_complete nulls found in null check
# kyc_complete is boolean — null means not recorded
# Safe to fill with 0 (assume not verified if not recorded)
modeling_data = modeling_data.fillna({"kyc_complete": 0})

# Verify
null_after = modeling_data.filter(
    F.col("kyc_complete").isNull()
).count()
print(f"kyc_complete nulls after fill: {null_after} ✅")

kyc_complete nulls after fill: 0 ✅


In [8]:
print("CATEGORY 5 — ENGINEERED RATIO FEATURES\n")

modeling_data = modeling_data.withColumn(
    # Log transform of income — reduces skew, better for linear models
    "log_income",
    F.round(F.log1p(F.col("monthly_income")), 4)
).withColumn(
    # Log transform of target — right skew requires this
    "log_clv",
    F.round(F.log1p(F.col("clv_next_12months")), 4)
).withColumn(
    # Income utilization — how much of income is spent annually
    "income_utilization",
    F.round(
        F.col("total_annual_debit") /
        (F.col("monthly_income") * 12 + F.lit(1.0)),
        4
    )
).withColumn(
    # Portfolio to income ratio
    "portfolio_to_income",
    F.round(
        F.col("total_portfolio_value") /
        (F.col("monthly_income") + F.lit(1.0)),
        2
    )
).withColumn(
    # Q4 to Q1 spend ratio — seasonal spending behavior
    "q4_q1_ratio",
    F.round(
        F.col("q4_spend") /
        (F.col("q1_spend") + F.lit(1.0)),
        4
    )
).withColumn(
    # Loan burden — total loan outstanding vs income
    "loan_to_income",
    F.round(
        (F.col("home_loan_value") + F.col("personal_loan_value")) /
        (F.col("monthly_income") + F.lit(1.0)),
        2
    )
)

print(f"After engineered features: {len(modeling_data.columns)} columns")
print("\nNew ratio feature distributions:")
modeling_data.select(
    F.round(F.avg("log_income"),4).alias("avg_log_income"),
    F.round(F.avg("log_clv"),4).alias("avg_log_clv"),
    F.round(F.avg("income_utilization"),4).alias("avg_income_util"),
    F.round(F.avg("portfolio_to_income"),2).alias("avg_portfolio_income"),
    F.round(F.avg("loan_to_income"),2).alias("avg_loan_income"),
).show()

# ── Target encoding for segment ───────────────────────────────
print("Target encoding segment by mean CLV...")
segment_means = modeling_data.groupBy("segment").agg(
    F.mean("clv_next_12months").alias("segment_mean_clv")
)
modeling_data = modeling_data.join(
    segment_means, on="segment", how="left"
)
print("  segment_mean_clv added ✅")

# ── Target encoding for city_tier ────────────────────────────
tier_means = modeling_data.groupBy("city_tier").agg(
    F.mean("clv_next_12months").alias("tier_mean_clv")
)
modeling_data = modeling_data.join(
    tier_means, on="city_tier", how="left"
)
print("  tier_mean_clv added ✅")

print(f"\nFinal modeling dataset: {modeling_data.count():,} rows "
      f"× {len(modeling_data.columns)} columns")

CATEGORY 5 — ENGINEERED RATIO FEATURES

After engineered features: 60 columns

New ratio feature distributions:
+--------------+-----------+---------------+--------------------+---------------+
|avg_log_income|avg_log_clv|avg_income_util|avg_portfolio_income|avg_loan_income|
+--------------+-----------+---------------+--------------------+---------------+
|       11.2573|     8.3245|         3.8558|               18.35|           7.17|
+--------------+-----------+---------------+--------------------+---------------+

Target encoding segment by mean CLV...
  segment_mean_clv added ✅
  tier_mean_clv added ✅

Final modeling dataset: 50,000 rows × 62 columns


In [9]:
print("SAVING FINAL MODELING DATASET\n")

PROCESSED = "../data/processed"

modeling_pd = modeling_data.toPandas()

print(f"Final dataset shape: {modeling_pd.shape}")
print(f"  Rows    : {len(modeling_pd):,}")
print(f"  Columns : {len(modeling_pd.columns)}")

modeling_pd.to_csv(
    f"{PROCESSED}/modeling_data.csv",
    index=False
)
print(f"\n✅ modeling_data.csv saved")

exclude_cols = [
    "customer_id","segment","city_tier","city",
    "gender","clv_bucket","clv_next_12months","log_clv"
]
feature_cols = [
    c for c in modeling_pd.columns
    if c not in exclude_cols
    and modeling_pd[c].dtype != object
]

import pickle
with open(f"{PROCESSED}/feature_cols.pkl","wb") as f:
    pickle.dump(feature_cols, f)

print(f"✅ feature_cols.pkl saved ({len(feature_cols)} features)")

print(f"\nTarget variable (clv_next_12months):")
print(f"  Mean   : ₹{modeling_pd['clv_next_12months'].mean():>12,.2f}")
print(f"  Median : ₹{modeling_pd['clv_next_12months'].median():>12,.2f}")
print(f"  Std    : ₹{modeling_pd['clv_next_12months'].std():>12,.2f}")
print(f"  Min    : ₹{modeling_pd['clv_next_12months'].min():>12,.2f}")
print(f"  Max    : ₹{modeling_pd['clv_next_12months'].max():>12,.2f}")

# Only print log_clv stats if the column exists
if "log_clv" in modeling_pd.columns:
    print(f"\nLog CLV distribution:")
    print(f"  Mean   : {modeling_pd['log_clv'].mean():>8.4f}")
    print(f"  Std    : {modeling_pd['log_clv'].std():>8.4f}")
    print(f"  Min    : {modeling_pd['log_clv'].min():>8.4f}")
    print(f"  Max    : {modeling_pd['log_clv'].max():>8.4f}")
else:
    print("\n  log_clv not in dataset — added in Cell 8")

print(f"\nFeature list ({len(feature_cols)} features):")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:>2}. {col}")

txn.unpersist()
print("\nCache cleared ✅")

SAVING FINAL MODELING DATASET



Final dataset shape: (50000, 62)
  Rows    : 50,000
  Columns : 62

✅ modeling_data.csv saved
✅ feature_cols.pkl saved (49 features)

Target variable (clv_next_12months):
  Mean   : ₹   44,628.94
  Median : ₹    8,992.96
  Std    : ₹  115,561.83
  Min    : ₹        0.00
  Max    : ₹  900,000.00

Log CLV distribution:
  Mean   :   8.3245
  Std    :   3.1612
  Min    :   0.0000
  Max    :  13.7102

Feature list (49 features):
   1. age
   2. monthly_income
   3. cibil_score
   4. digital_score
   5. account_age_months
   6. kyc_complete
   7. pan_linked
   8. aadhaar_linked
   9. rm_assigned
  10. is_nri
  11. total_txns
  12. total_annual_debit
  13. avg_debit_amount
  14. avg_txn_amount
  15. std_txn_amount
  16. max_txn_amount
  17. unique_categories
  18. unique_txn_types
  19. active_months
  20. q4_spend
  21. q1_spend
  22. spend_last_3m
  23. spend_first_9m
  24. spend_trend_ratio
  25. avg_monthly_debit
  26. std_monthly_debit
  27. peak_month_spend
  28. zero_spend_months
  29.